# 🦙 QuarkLlama: From Scratch to Chat Assistant (0.5B Parameters)
### *The Complete 2-Phase Training Pipeline: Pre-Training & Supervised Fine-Tuning (SFT)*

---

## 📖 Overview & The 2-Phase Lifecycle
How does a raw neural network transform into an AI that answers questions like ChatGPT and TinyLlama-Chat? **It requires a 2-Phase process**:

```
┌─────────────────────────────────────────────────────────────────────────┐
│ PHASE 1: Pre-Training (The "School" Phase)                              │
│ • Trained on raw continuous text (TinyStories)                          │
│ • Learns grammar, English words, sentence structure, and facts          │
│ • Result: Base Model (Completes text, but CANNOT answer questions yet)  │
│ • Checkpoint: quarkllama_base.pt                                        │
└────────────────────────────────────┬────────────────────────────────────┘
                                     │
                                     ▼
┌─────────────────────────────────────────────────────────────────────────┐
│ PHASE 2: Supervised Fine-Tuning / SFT (The "Job Training" Phase)        │
│ • Trained on Question & Answer pairs (Alpaca / Dolly-15k)               │
│ • Uses ChatML templates (<|user|> and <|assistant|>)                   │
│ • Uses Masked Loss (only learns from the assistant's answer)            │
│ • Result: Chat Assistant (Directly answers questions and stops cleanly) │
│ • Checkpoint: quarkllama_chat.pt                                        │
└─────────────────────────────────────────────────────────────────────────┘
```

---

### 🏆 Modern Architectural Innovations in QuarkLlama:
1. **RMSNorm** — Replaces LayerNorm for faster, numerically stable forward passes without mean calculation.
2. **RoPE (Rotary Position Embeddings)** — Encodes word order using 2D complex plane rotations instead of static vectors.
3. **GQA (Grouped-Query Attention)** — 16 Query heads share 4 Key/Value heads to slash memory bandwidth by 75%.
4. **SwiGLU** — Gated linear unit FFN with SiLU activation for rich non-linear capacity.

## 🛠️ Step 1: Environment Setup & Hardware Verification

### What happens in this step?
We detect the available GPU (NVIDIA T4, L4, or A100), check VRAM, and install required libraries (`datasets`, `transformers`, `accelerate`).

In [ ]:
# Install dependencies
!pip install -q datasets transformers accelerate tiktoken

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import os
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detected: {gpu_name} ({total_mem:.2f} GB VRAM)")
    device = torch.device("cuda")
else:
    print("⚠️ No GPU detected! Please go to Runtime -> Change runtime type -> T4 GPU.")
    device = torch.device("cpu")

## 📐 Step 2: QuarkLlama Architectural Configuration

### Parameter Math (~501 Million / 0.501 Billion Parameters):
- **Vocabulary ($V$)**: 32,000 tokens
- **Hidden Dimension ($d$)**: 1,152
- **Layers ($L$)**: 28 transformer blocks
- **Query Heads**: 16 (Head dimension $d_k = 1152 / 16 = 72$)
- **KV Heads**: 4 (Grouped-Query Attention 4:1 ratio)
- **FFN Dimension ($d_{ff}$)**: 3,456 ($3 \times d$ using SwiGLU)

$$\text{Total Parameters} = \mathbf{501,121,152} \text{ parameters (meets } \ge 0.5\text{B requirement)}!$$

In [ ]:
@dataclass
class QuarkLlamaConfig:
    vocab_size: int = 32000              # 32k BPE vocabulary
    hidden_size: int = 1152              # Hidden dimension (d_model)
    intermediate_size: int = 3456        # SwiGLU FFN dimension (3 * hidden_size)
    num_hidden_layers: int = 28          # 28 Transformer layers
    num_attention_heads: int = 16        # 16 Query heads
    num_key_value_heads: int = 4         # 4 KV heads (GQA 4:1)
    max_position_embeddings: int = 1024  # Max sequence context
    rope_theta: float = 10000.0          # Base frequency for RoPE
    rms_norm_eps: float = 1e-5           # Epsilon for RMSNorm stability
    tie_word_embeddings: bool = False    # Untied LM head for max capacity

# Mode toggle: Set to 'FULL_0.5B' for true 0.5B model, or 'QUICK_TEST' for rapid 2-minute testing
TRAINING_MODE = "FULL_0.5B"

if TRAINING_MODE == "QUICK_TEST":
    config = QuarkLlamaConfig(
        hidden_size=512,
        intermediate_size=1536,
        num_hidden_layers=6,
        num_attention_heads=8,
        num_key_value_heads=2,
        max_position_embeddings=512
    )
    print("⚡ Configured in QUICK_TEST mode (~35M params).")
else:
    config = QuarkLlamaConfig()
    print("🚀 Configured in FULL_0.5B mode (501 Million parameters).")

## 🧩 Step 3: Modern Transformer Architecture Modules

Here we implement the 4 core pillars of the TinyLlama / LLaMA architecture:
1. **RMSNorm**: Root Mean Square Normalization (faster than LayerNorm without mean tracking).
2. **RoPE (Rotary Position Embeddings)**: Rotates Query and Key vectors in complex 2D space.
3. **Grouped-Query Attention (GQA)**: 4:1 sharing of KV heads to reduce VRAM consumption.
4. **SwiGLU FFN**: $\text{SwiGLU}(x) = (\text{SiLU}(x W_{gate}) \odot x W_{up}) W_{down}$.

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization"""
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output = self._norm(x.float()).type_as(x)
        return output * self.weight


def precompute_rope_frequencies(dim: int, max_seq_len: int, theta: float = 10000.0) -> torch.Tensor:
    """Precompute sinusoidal frequencies for RoPE: e^(i * m * theta)"""
    assert dim % 2 == 0
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)


def apply_rotary_emb(xq: torch.Tensor, xk: torch.Tensor, freqs_cis: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """Apply 2D complex rotation to Query and Key tensors"""
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(2)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)


class GroupedQueryAttention(nn.Module):
    """Grouped-Query Attention (GQA) with FlashAttention backend"""
    def __init__(self, config: QuarkLlamaConfig):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.num_kv_groups = self.num_heads // self.num_kv_heads

        self.q_proj = nn.Linear(config.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, config.hidden_size, bias=False)

    def forward(self, x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
        bsz, seqlen, _ = x.shape

        xq = self.q_proj(x).view(bsz, seqlen, self.num_heads, self.head_dim)
        xk = self.k_proj(x).view(bsz, seqlen, self.num_kv_heads, self.head_dim)
        xv = self.v_proj(x).view(bsz, seqlen, self.num_kv_heads, self.head_dim)

        xq, xk = apply_rotary_emb(xq, xk, freqs_cis)

        xq = xq.transpose(1, 2)
        xk = xk.transpose(1, 2)
        xv = xv.transpose(1, 2)

        if self.num_kv_groups > 1:
            xk = xk.repeat_interleave(self.num_kv_groups, dim=1)
            xv = xv.repeat_interleave(self.num_kv_groups, dim=1)

        # FlashAttention Causal Attention via PyTorch SDPA
        output = F.scaled_dot_product_attention(xq, xk, xv, is_causal=True)
        output = output.transpose(1, 2).contiguous().view(bsz, seqlen, -1)
        return self.o_proj(output)


class SwiGLUFFN(nn.Module):
    """SwiGLU Gated Feed-Forward Network"""
    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class QuarkLlamaBlock(nn.Module):
    """Single Transformer Decoder Layer"""
    def __init__(self, config: QuarkLlamaConfig):
        super().__init__()
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.self_attn = GroupedQueryAttention(config)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.mlp = SwiGLUFFN(config.hidden_size, config.intermediate_size)

    def forward(self, x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.input_layernorm(x), freqs_cis)
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x


class QuarkLlamaForCausalLM(nn.Module):
    """Complete QuarkLlama Language Model"""
    def __init__(self, config: QuarkLlamaConfig):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([QuarkLlamaBlock(config) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        if config.tie_word_embeddings:
            self.lm_head.weight = self.embed_tokens.weight

        head_dim = config.hidden_size // config.num_attention_heads
        freqs_cis = precompute_rope_frequencies(head_dim, config.max_position_embeddings, config.rope_theta)
        self.register_buffer("freqs_cis", freqs_cis, persistent=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None):
        bsz, seqlen = input_ids.shape
        x = self.embed_tokens(input_ids)
        freqs_cis = self.freqs_cis[:seqlen].to(x.device)

        for layer in self.layers:
            x = layer(x, freqs_cis)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            # ignore_index = -100 allows masked loss for Phase 2 Instruction Tuning!
            loss = F.cross_entropy(
                shift_logits.view(-1, self.config.vocab_size),
                shift_labels.view(-1),
                ignore_index=-100
            )

        return logits, loss

## 🔍 Step 4: Model Instantiation & Parameter Verification

In [ ]:
model = QuarkLlamaForCausalLM(config).to(device)
total_params = sum(p.numel() for p in model.parameters())
print("=" * 60)
print(f"🌟 Model Name: QuarkLlama")
print(f"📊 Total Parameters:     {total_params:,} ({total_params / 1e9:.3f} Billion)")
print(f"🧱 Hidden Layers:        {config.num_hidden_layers}")
print(f"📏 Hidden Dimension:     {config.hidden_size}")
print(f"👥 Query / KV Heads:     {config.num_attention_heads} / {config.num_key_value_heads}")
print("=" * 60)
assert total_params > 500_000_000 if TRAINING_MODE == 'FULL_0.5B' else True, "Model should exceed 0.5B!"

## 🔤 Step 5: Tokenizer Setup

We load a standard 32k BPE tokenizer (TinyLlama/LLaMA vocabulary) and verify special tokens.

In [ ]:
from transformers import AutoTokenizer

tokenizer_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Loading tokenizer: {tokenizer_name}...")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab Size: {tokenizer.vocab_size}")
print(f"EOS Token:  {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

# 🏁 PHASE 1: Pre-Training (Next-Token Language Modeling)

### What happens in Phase 1?
The model reads raw, continuous stories from the **`roneneldan/TinyStories`** dataset.
At every step, it tries to predict word $t+1$ given words $1, \dots, t$.

### What QuarkLlama learns here:
- Grammar, English vocabulary, word relationships, and sentence flow.
- **Note**: At the end of Phase 1, the model is a **raw text completer** (it will complete stories, but doesn't know how to follow Q&A instructions yet).

In [ ]:
from datasets import load_dataset
from torch.utils.data import IterableDataset, DataLoader

class StreamingPretrainDataset(IterableDataset):
    """Streams raw text and packs tokens into fixed 512-length sequences"""
    def __init__(self, dataset_stream, tokenizer, seq_len=512):
        self.dataset_stream = dataset_stream
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __iter__(self):
        buffer = []
        for example in self.dataset_stream:
            text = example.get("text", "")
            if not text.strip():
                continue
            tokens = self.tokenizer.encode(text) + [self.tokenizer.eos_token_id]
            buffer.extend(tokens)
            while len(buffer) >= self.seq_len:
                chunk = buffer[:self.seq_len]
                buffer = buffer[self.seq_len:]
                yield torch.tensor(chunk, dtype=torch.long)

print("Streaming TinyStories dataset for Phase 1 Pre-training...")
raw_pretrain_data = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
pretrain_dataset = StreamingPretrainDataset(raw_pretrain_data, tokenizer, seq_len=512)
pretrain_loader = DataLoader(pretrain_dataset, batch_size=4 if TRAINING_MODE == 'FULL_0.5B' else 8)

In [ ]:
# Phase 1 Hyperparameters (Pre-Training: ~2.5 - 3 hours)
PHASE1_MAX_HOURS = 2.8      # Pre-training time budget
PHASE1_STEPS = 3500
GRAD_ACCUM_STEPS = 8
LR_PRETRAIN = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PRETRAIN, betas=(0.9, 0.95), weight_decay=0.1)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

print("🚀 Starting Phase 1: Pre-training (Next-Token Prediction)...")
model.train()
start_time = time.time()
step = 0
accum_loss = 0.0
optimizer.zero_grad(set_to_none=True)

os.makedirs("./checkpoints", exist_ok=True)

for batch_idx, batch_tokens in enumerate(pretrain_loader):
    elapsed_hours = (time.time() - start_time) / 3600.0
    if elapsed_hours >= PHASE1_MAX_HOURS or step >= PHASE1_STEPS:
        print(f"\n🔔 Phase 1 complete! Elapsed: {elapsed_hours:.2f}h, Steps: {step}.")
        break

    inputs = batch_tokens.to(device)
    labels = inputs.clone()  # In pretraining, labels are simply the inputs (shifted in forward)

    with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
        _, loss = model(inputs, labels=labels)
        loss = loss / GRAD_ACCUM_STEPS

    scaler.scale(loss).backward()
    accum_loss += loss.item() * GRAD_ACCUM_STEPS

    if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        step += 1

        if step % 20 == 0 or step == 1:
            ppl = math.exp(min(accum_loss, 20))
            print(f"[Pre-train Step {step:04d}] Loss: {accum_loss:.4f} | Perplexity: {ppl:.2f} | Elapsed: {elapsed_hours:.2f}h")
            accum_loss = 0.0

# Save Base Model Checkpoint
base_model_path = "./checkpoints/quarkllama_base.pt"
torch.save({'model_state_dict': model.state_dict(), 'config': config}, base_model_path)
print(f"💾 Phase 1 Base Model saved to {base_model_path}!")

# 🎯 PHASE 2: Supervised Fine-Tuning (SFT / Instruction Tuning)

### What happens in Phase 2?
Now we teach QuarkLlama to act as an **AI Assistant that answers questions**.

### The 2 Secret Techniques of SFT:
1. **The ChatML Template**:
   Every training example is formatted like this:
   ```text
   <|user|>
   What is the capital of France?
   <|assistant|>
   The capital of France is Paris.<|im_end|>
   ```
2. **Masked Loss (`ignore_index = -100`)**:
   - In pre-training, we computed loss on every single token.
   - In instruction tuning, **we set label = -100 for all User tokens**.
   - PyTorch's `F.cross_entropy(..., ignore_index=-100)` **completely ignores** tokens with ID -100.
   - The model is **only penalized for mistakes in the Assistant's answer**!
   - This trains QuarkLlama to generate accurate answers rather than echoing the question.

In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

# Load Databricks Dolly 15k or Alpaca Instruction Dataset
print("Loading Instruction Q&A Dataset (databricks/databricks-dolly-15k)...")
instruct_data = load_dataset("databricks/databricks-dolly-15k", split="train")

class InstructionQADataset(Dataset):
    """
    Formats Question-Answer pairs with Masked Loss on the user prompt.
    """
    def __init__(self, raw_data, tokenizer, max_len=512):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.samples = []

        print("Formatting Q&A pairs with ChatML template and Masked Loss...")
        for item in raw_data:
            instruction = item.get("instruction", "").strip()
            context = item.get("context", "").strip()
            response = item.get("response", "").strip()
            
            if not instruction or not response:
                continue
                
            user_msg = f"{instruction}\n{context}".strip() if context else instruction
            
            # ChatML prompt structure
            prompt_part = f"<|user|>\n{user_msg}\n<|assistant|>\n"
            answer_part = f"{response}</s>"
            
            prompt_ids = self.tokenizer.encode(prompt_part)
            answer_ids = self.tokenizer.encode(answer_part)
            
            input_ids = prompt_ids + answer_ids
            if len(input_ids) > self.max_len:
                continue  # skip oversized items
                
            # MASKED LOSS: Set -100 for all prompt tokens so loss is ONLY computed on the answer!
            labels = [-100] * len(prompt_ids) + answer_ids
            
            # Pad to max_len
            pad_len = self.max_len - len(input_ids)
            input_ids += [self.tokenizer.pad_token_id] * pad_len
            labels += [-100] * pad_len
            
            self.samples.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)
            })
            
            if len(self.samples) >= 3000:  # Curated subset for 45-minute SFT
                break
                
        print(f"✅ Successfully prepared {len(self.samples)} high-quality Instruction Q&A pairs!")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

sft_dataset = InstructionQADataset(instruct_data, tokenizer, max_len=512)
sft_loader = DataLoader(sft_dataset, batch_size=4, shuffle=True)

In [ ]:
# Phase 2 Hyperparameters (SFT Instruction Tuning: ~45 - 60 minutes)
PHASE2_EPOCHS = 2
LR_SFT = 5e-5               # Lower learning rate for fine-tuning to preserve learned knowledge
SFT_GRAD_ACCUM = 4

optimizer_sft = torch.optim.AdamW(model.parameters(), lr=LR_SFT, betas=(0.9, 0.95), weight_decay=0.01)

print("🚀 Starting Phase 2: Supervised Fine-Tuning (SFT / Chat Tuning)...")
model.train()
start_time_sft = time.time()
sft_step = 0

for epoch in range(PHASE2_EPOCHS):
    running_loss = 0.0
    for batch_idx, batch in enumerate(sft_loader):
        inputs = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            _, loss = model(inputs, labels=labels)
            loss = loss / SFT_GRAD_ACCUM

        scaler.scale(loss).backward()
        running_loss += loss.item() * SFT_GRAD_ACCUM

        if (batch_idx + 1) % SFT_GRAD_ACCUM == 0:
            scaler.unscale_(optimizer_sft)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer_sft)
            scaler.update()
            optimizer_sft.zero_grad(set_to_none=True)
            sft_step += 1

            if sft_step % 10 == 0:
                print(f"[SFT Epoch {epoch+1}/{PHASE2_EPOCHS} | Step {sft_step:03d}] Answer Loss: {running_loss:.4f}")
                running_loss = 0.0

# Save Final Chat Model Checkpoint
chat_model_path = "./checkpoints/quarkllama_chat.pt"
torch.save({'model_state_dict': model.state_dict(), 'config': config}, chat_model_path)
print(f"🎉 Both Phases Complete! Final QuarkLlama Chat Model saved to {chat_model_path}!")

## 💬 Step 6: Interactive Q&A Chat Engine

Now that QuarkLlama has undergone **both Phase 1 (Pre-training)** and **Phase 2 (Instruction Tuning)**, it understands the ChatML format!

When you call `ask_quarkllama("What is photosynthesis?")`:
1. It formats the prompt as `<|user|>\nWhat is photosynthesis?\n<|assistant|>\n`
2. QuarkLlama predicts the answer token by token.
3. It stops when it generates the stop token `</s>`.

In [ ]:
@torch.no_grad()
def ask_quarkllama(
    question: str,
    max_new_tokens: int = 150,
    temperature: float = 0.7,
    top_p: float = 0.9,
    top_k: int = 40,
    repetition_penalty: float = 1.15
) -> str:
    """
    Ask QuarkLlama a question and get a direct answer (like ChatGPT)!
    """
    model.eval()
    
    # Wrap in ChatML format that QuarkLlama learned in Phase 2
    prompt = f"<|user|>\n{question.strip()}\n<|assistant|>\n"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    prompt_len = input_ids.shape[1]
    
    curr_tokens = input_ids.clone()
    for _ in range(max_new_tokens):
        idx_cond = curr_tokens if curr_tokens.size(1) <= config.max_position_embeddings else curr_tokens[:, -config.max_position_embeddings:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]  # Next-token logits
        
        # Repetition penalty
        for token_id in set(curr_tokens[0].tolist()):
            if logits[0, token_id] > 0:
                logits[0, token_id] /= repetition_penalty
            else:
                logits[0, token_id] *= repetition_penalty
                
        # Temperature scaling
        logits = logits / max(temperature, 1e-5)
        
        # Top-K
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')
            
        # Top-P (Nucleus)
        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
            logits[indices_to_remove] = -float('Inf')
            
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        # If stop token emitted, finish answer
        if next_token.item() == tokenizer.eos_token_id:
            break
            
        curr_tokens = torch.cat((curr_tokens, next_token), dim=1)

    # Decode only the assistant's generated answer (strip the prompt)
    generated_ids = curr_tokens[0, prompt_len:].tolist()
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return answer

## 🧪 Step 7: Test Asking Questions to QuarkLlama!

Type your questions below and watch QuarkLlama generate direct answers!

In [ ]:
questions = [
    "What is the difference between a star and a planet?",
    "Why do birds sing in the morning?",
    "Give me 3 tips to stay healthy."
]

print("✨ --- QuarkLlama Q&A Showcase --- ✨\n")
for q in questions:
    ans = ask_quarkllama(q, max_new_tokens=100, temperature=0.7)
    print(f"❓ Question: {q}")
    print(f"💡 QuarkLlama Answer: {ans}")
    print("-" * 70)

## 💬 Interactive Prompt: Ask Your Own Question

In [ ]:
# Put your own question here:
my_question = "Explain gravity in simple words for a 10-year-old child."

print(f"🧑 You: {my_question}")
response = ask_quarkllama(my_question, max_new_tokens=120, temperature=0.7)
print(f"🦙 QuarkLlama: {response}")